# LightGBM + 三个神经网络：概率融合

在 notebook 内训练四个模型，按官方 API 逐批累积访问历史，平均评分概率后按 SMAPE+1 做预测。挂载官方比赛数据，关闭 Internet。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from torch import nn
from lightgbm import LGBMClassifier

sys.path.append('/kaggle/input/amp-pd')
torch.set_num_threads(1)
DATA_DIR = '/kaggle/input/amp-pd'
TARGETS = [f'updrs_{i}' for i in range(1, 5)]
KEYS = ['patient_id', 'visit_month']
MILESTONES = (6, 18)
SMALL = ['P05060', 'O15240', 'P17174']
SEED = 4486
EPOCHS = 100
PARAMS = dict(n_estimators=150, learning_rate=.03, num_leaves=7, max_depth=3,
              min_child_samples=50, reg_lambda=10.0, random_state=SEED,
              n_jobs=1, verbosity=-1, deterministic=True, force_col_wise=True)


In [ ]:
def read_data(data_dir):
    data_dir = Path(data_dir)
    clinical = pd.read_csv(data_dir / 'train_clinical_data.csv')
    omics = pd.concat([pd.read_csv(data_dir / name, usecols=KEYS) for name in
                       ('train_proteins.csv', 'train_peptides.csv')]).drop_duplicates()
    return clinical, omics

def history_codes(requests, visits):
    """Milestone visits are visible only at or after their month."""
    result = np.zeros(len(requests), dtype=int)
    for i, milestone in enumerate(MILESTONES):
        seen = set(visits.loc[visits.visit_month.eq(milestone), 'patient_id'])
        digit = np.where(requests.visit_month.to_numpy() < milestone, -1,
                         requests.patient_id.isin(seen).to_numpy().astype(int))
        result += (digit + 1) * 3 ** i
    return result

def training_rows(clinical, omics, stream='union'):
    origins = omics.drop_duplicates().copy()
    requests = origins.merge(pd.DataFrame({'horizon': [0, 6, 12, 24]}), how='cross')
    requests['target_month'] = requests.visit_month + requests.horizon
    labels = clinical.melt(id_vars=KEYS, value_vars=TARGETS, var_name='target',
                           value_name='rating').dropna(subset=['rating'])
    labels = labels.rename(columns={'visit_month': 'target_month'})
    rows = requests.merge(labels, on=['patient_id', 'target_month'], validate='many_to_many')
    if rows.duplicated(['patient_id', 'visit_month', 'target', 'horizon']).any():
        raise ValueError('Duplicate training requests; check clinical visit keys')
    visits = pd.concat([clinical[KEYS], omics]).drop_duplicates() if stream == 'union' else omics
    rows['history'] = history_codes(rows, visits)
    return rows

def ranking(clinical, wide):
    """Prior experiment's mean absolute month-adjusted Spearman, current scores."""
    current = clinical.merge(wide.reset_index(), on=KEYS, how='inner', validate='one_to_one')
    records = []
    for target in TARGETS:
        for protein in wide.columns:
            v = current[[protein, target, 'visit_month', 'patient_id']].dropna()
            if len(v) < 100 or v.patient_id.nunique() < 50 or v[protein].nunique() < 2 or v[target].nunique() < 2:
                continue
            x, y, month = (v[c].rank().to_numpy() for c in (protein, target, 'visit_month'))
            control = np.column_stack([np.ones(len(v)), month])
            x = x - control @ np.linalg.lstsq(control, x, rcond=None)[0]
            y = y - control @ np.linalg.lstsq(control, y, rcond=None)[0]
            corr = np.corrcoef(x, y)[0, 1]
            if np.isfinite(corr):
                records.append((protein, target, abs(corr), len(v), v.patient_id.nunique()))
    scores = pd.DataFrame(records, columns=['protein', 'target', 'abs_correlation', 'visits', 'patients'])
    order = scores.groupby('protein').agg(score=('abs_correlation', 'mean'), n_targets=('target', 'nunique'),
                                            min_visits=('visits', 'min'), min_patients=('patients', 'min'))
    return order.loc[order.n_targets.eq(4)].sort_values(['score', 'protein'], ascending=[False, True]).reset_index()

def history_features(rows, clinical, omics, proteins):
    """Causal visit and protein history; no clinical scores are inputs."""
    union = pd.concat([clinical[KEYS], omics]).drop_duplicates()
    pvisits = proteins[KEYS].drop_duplicates()
    visit_map = {pid: sorted(group.visit_month.unique()) for pid, group in union.groupby('patient_id')}
    protein_map = {pid: sorted(group.visit_month.unique()) for pid, group in pvisits.groupby('patient_id')}
    unique = rows[KEYS].drop_duplicates().copy()
    features = []
    for row in unique.itertuples(index=False):
        history = [m for m in visit_map.get(row.patient_id, []) if m <= row.visit_month]
        measured = [m for m in protein_map.get(row.patient_id, []) if m <= row.visit_month]
        features.append([int(6 in history), int(18 in history), int(bool(measured)),
                         int(row.visit_month in measured)])
    unique[['seen_6', 'seen_18', 'protein_known', 'protein_now']] = np.asarray(features)
    out = rows[KEYS].merge(unique, on=KEYS, how='left', validate='many_to_one')
    out[['visit_month', 'horizon', 'target_month']] = rows[['visit_month', 'horizon', 'target_month']].to_numpy()
    return out[['visit_month', 'horizon', 'target_month', 'seen_6', 'seen_18', 'protein_known', 'protein_now']]

def decision(prob):
    assert np.isfinite(prob).all() and (prob >= 0).all()
    np.testing.assert_allclose(prob.sum(axis=1), 1, atol=1e-5)
    classes = np.arange(prob.shape[1])
    loss = 200 * abs(classes[:, None] - classes) / (classes[:, None] + classes + 2)
    return classes[(prob @ loss).argmin(axis=1)]

def spread_prob(prob, classes, nclasses):
    result = np.zeros((len(prob), nclasses), dtype=np.float32)
    result[:, np.asarray(classes, dtype=int)] = prob
    return result

class HistoryMLP(nn.Module):
    def __init__(self, inputs, nclasses):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(inputs, 24), nn.ReLU(), nn.Linear(24, 16),
                                 nn.ReLU(), nn.Linear(16, nclasses))
    def forward(self, base, protein, mask):
        return self.net(base)

class SmallMLP(nn.Module):
    def __init__(self, inputs, nclasses, nproteins):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(inputs + 2*nproteins, 24), nn.ReLU(),
                                 nn.Linear(24, 16), nn.ReLU(), nn.Linear(16, nclasses))
    def forward(self, base, protein, mask):
        return self.net(torch.cat([base, protein, mask], dim=1))

class SetMLP(nn.Module):
    def __init__(self, inputs, nclasses, nproteins):
        super().__init__()
        self.embedding = nn.Embedding(nproteins, 3)
        self.encoder = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())
        self.net = nn.Sequential(nn.Linear(inputs+4, 24), nn.ReLU(), nn.Linear(24, nclasses))
    def forward(self, base, protein, mask):
        ids = torch.arange(protein.shape[1], device=protein.device)
        identity = self.embedding(ids).unsqueeze(0).expand(len(base), -1, -1)
        encoded = self.encoder(torch.cat([protein.unsqueeze(-1), identity], dim=-1))
        pooled = (encoded*mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        return self.net(torch.cat([base, pooled], dim=1))

In [ ]:

def prepare_protein(values):
    mask = np.isfinite(values).astype(np.float32)
    filled = np.nan_to_num(values, nan=0.0).astype(np.float32)
    return torch.from_numpy(filled), torch.from_numpy(mask)


def fit_neural(name, base, proteins, y, classes, seed):
    torch.manual_seed(seed)
    if name == 'nn_history': net = HistoryMLP(base.shape[1], len(classes))
    elif name == 'nn_small': net = SmallMLP(base.shape[1], len(classes), proteins.shape[1])
    else: net = SetMLP(base.shape[1], len(classes), proteins.shape[1])
    values, mask = prepare_protein(proteins)
    x = torch.from_numpy(base.astype(np.float32))
    labels = torch.from_numpy(np.searchsorted(classes, y).astype(np.int64))
    optimizer = torch.optim.AdamW(net.parameters(), lr=.01, weight_decay=.01)
    for _ in range(EPOCHS):
        net.train()
        loss = nn.functional.cross_entropy(net(x, values, mask), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return net.eval()


clinical, omics = read_data(DATA_DIR)
rows = training_rows(clinical, omics, 'union')
protein = pd.read_csv(Path(DATA_DIR) / 'train_proteins.csv', usecols=KEYS + ['UniProt', 'NPX'])
wide = protein.pivot(index=KEYS, columns='UniProt', values='NPX').sort_index()
train_patients = set(rows.patient_id)
train_wide = wide.loc[wide.index.get_level_values('patient_id').isin(train_patients)]
ranked = ranking(clinical[clinical.patient_id.isin(train_patients)], train_wide)
TOP10 = ranked.head(10).protein.tolist()
if len(TOP10) != 10:
    raise ValueError('Insufficient proteins for the trained Top 10 model')
protein_cols = list(dict.fromkeys(SMALL + TOP10))
log_train = np.log1p(train_wide[protein_cols])
protein_mean = log_train.mean()
protein_std = log_train.std(ddof=0).replace(0, 1)
normalized = ((np.log1p(wide[protein_cols]) - protein_mean) / protein_std).clip(-5, 5)
z = normalized.reindex(pd.MultiIndex.from_frame(rows[KEYS])).reset_index(drop=True)
features = history_features(rows, clinical, omics, protein)
base_mean, base_std = features.to_numpy().mean(0), features.to_numpy().std(0)
base_std[base_std == 0] = 1
base = np.clip((features.to_numpy() - base_mean) / base_std, -8, 8)
models = {}
for target in TARGETS:
    train = rows[rows.target.eq(target)]
    y = train.rating.to_numpy(dtype=int)
    classes = np.unique(y)
    lgbm = LGBMClassifier(**PARAMS)
    lgbm.fit(features.loc[train.index], y)
    networks = {}
    for offset, (name, cols) in enumerate([('nn_history', []), ('nn_small', SMALL), ('nn_set', TOP10)]):
        networks[name] = fit_neural(name, base[train.index],
                                    z.loc[train.index, cols].to_numpy(dtype=np.float32),
                                    y, classes, SEED + offset)
    models[target] = (lgbm, networks, classes)


In [ ]:

visit_state = set()
protein_state = set()
last_month = None


def predict_batch(test, test_peptides, test_proteins, sample_submission):
    global last_month
    sample = sample_submission.reset_index(drop=True)
    requests = sample.prediction_id.str.extract(
        r'^(?P<patient_id>\d+)_(?P<visit_month>\d+)_(?P<target>updrs_[1-4])_plus_(?P<horizon>0|6|12|24)_months$')
    if requests.isna().any().any():
        raise ValueError('Unexpected prediction_id format')
    for col in ('patient_id', 'visit_month', 'horizon'):
        requests[col] = requests[col].astype(int)
    requests['target_month'] = requests.visit_month + requests.horizon
    current = pd.concat([df[KEYS] for df in (test, test_peptides, test_proteins)], ignore_index=True).drop_duplicates()
    months = pd.concat([current.visit_month, requests.visit_month]).dropna().unique()
    if len(months) != 1:
        raise ValueError('Expected one visit month per API batch')
    month = int(months[0])
    if last_month is not None and month < last_month:
        raise ValueError('API batches must be in chronological order')
    last_month = month
    visit_state.update(map(tuple, current[KEYS].itertuples(index=False, name=None)))
    protein_state.update(map(tuple, test_proteins[KEYS].drop_duplicates().itertuples(index=False, name=None)))
    visits = pd.DataFrame(list(visit_state), columns=KEYS)
    pvisits = pd.DataFrame(list(protein_state), columns=KEYS)
    f = history_features(requests, visits, visits, pvisits)
    x = np.clip((f.to_numpy() - base_mean) / base_std, -8, 8)
    selected = test_proteins.loc[test_proteins.UniProt.isin(protein_cols), KEYS + ['UniProt','NPX']]
    current_wide = selected.pivot(index=KEYS, columns='UniProt', values='NPX').reindex(columns=protein_cols)
    zcur = ((np.log1p(current_wide) - protein_mean) / protein_std).clip(-5, 5)
    zcur = zcur.reindex(pd.MultiIndex.from_frame(requests[KEYS])).reset_index(drop=True)
    prediction = np.empty(len(requests), dtype=float)
    for target, group in requests.groupby('target', sort=False):
        idx = group.index.to_numpy()
        lgbm, nets, classes = models[target]
        nclasses = int(classes.max()) + 1
        probs = [spread_prob(lgbm.predict_proba(f.iloc[idx]), lgbm.classes_, nclasses)]
        for name, cols in [('nn_history', []), ('nn_small', SMALL), ('nn_set', TOP10)]:
            v, mask = prepare_protein(zcur.loc[idx, cols].to_numpy(dtype=np.float32))
            with torch.no_grad():
                logits = nets[name](torch.from_numpy(x[idx].astype(np.float32)), v, mask)
                p = torch.softmax(logits, dim=1).numpy()
            probs.append(spread_prob(p, classes, nclasses))
        prediction[idx] = decision(np.mean(probs, axis=0))
    result = sample_submission.copy()
    result['rating'] = prediction
    return result


In [ ]:
import amp_pd_peptide
env = amp_pd_peptide.make_env()
iter_test = env.iter_test()

In [ ]:
for (test, test_peptides, test_proteins, sample_submission) in iter_test:
    prediction = predict_batch(test, test_peptides, test_proteins, sample_submission)
    env.predict(prediction)